In [2]:
import duckdb 

In [3]:
conn=duckdb.connect(r"D:\SEC.gov project\Analysis Project\credit_risk.db")

IOException: IO Error: Cannot open file "D:\SEC.gov project\Analysis Project\credit_risk.db": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Users\rishabh hinduja\AppData\Local\Programs\Python\Python313\python.exe (PID 7216)

In [7]:
conn.sql(""" 
SELECT COUNT(*)
FROM sub 
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2578 │
└──────────────┘



In [11]:
conn.sql("""
PRAGMA table_info('sub')
""").show()

┌───────┬─────────────┬─────────┬─────────┬────────────┬─────────┐
│  cid  │    name     │  type   │ notnull │ dflt_value │   pk    │
│ int32 │   varchar   │ varchar │ boolean │  varchar   │ boolean │
├───────┼─────────────┼─────────┼─────────┼────────────┼─────────┤
│     0 │ adsh        │ VARCHAR │ false   │ NULL       │ false   │
│     1 │ cik         │ VARCHAR │ false   │ NULL       │ false   │
│     2 │ name        │ VARCHAR │ false   │ NULL       │ false   │
│     3 │ sic         │ VARCHAR │ false   │ NULL       │ false   │
│     4 │ changed     │ VARCHAR │ false   │ NULL       │ false   │
│     5 │ afs         │ VARCHAR │ false   │ NULL       │ false   │
│     6 │ wksi        │ VARCHAR │ false   │ NULL       │ false   │
│     7 │ fye         │ VARCHAR │ false   │ NULL       │ false   │
│     8 │ form        │ VARCHAR │ false   │ NULL       │ false   │
│     9 │ period      │ VARCHAR │ false   │ NULL       │ false   │
│    10 │ fy          │ VARCHAR │ false   │ NULL       │ false

In [13]:
conn.sql("""
SUMMARIZE sub
""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,adsh,VARCHAR,0000014195-19-000008,0001968915-25-000066,2724,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
1,cik,VARCHAR,1005229,97216,145,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
2,name,VARCHAR,10X CAPITAL VENTURE ACQUISITION CORP,YONG BAI CHAO NEW RETAIL CORP,195,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
3,sic,VARCHAR,3510,3716,9,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
4,changed,VARCHAR,19600201,20250731,97,<NA>,<NA>,<NA>,<NA>,<NA>,2578,35.92
5,afs,VARCHAR,1-LAF,4-NON,3,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
6,wksi,VARCHAR,0,1,2,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
7,fye,VARCHAR,0228,1231,12,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
8,form,VARCHAR,10-K,10-Q,2,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00
9,period,VARCHAR,20171231,20251130,78,<NA>,<NA>,<NA>,<NA>,<NA>,2578,0.00


In [5]:
conn.sql("""
         SELECT adsh, COUNT(*)
         FROM sub
         GROUP BY adsh
         HAVING COUNT(*) > 1
         """).show()

┌─────────┬──────────────┐
│  adsh   │ count_star() │
│ varchar │    int64     │
└─────────┴──────────────┘
          0 rows        



adsh verified, no duplicates 

In [6]:
conn.sql(""" 
         SELECT 
         cik,
         COUNT(DISTINCT name)
         FROM sub
         GROUP BY cik
         HAVING COUNT(DISTINCT name) > 1
         """)

┌─────────┬────────────────────────┐
│   cik   │ count(DISTINCT "name") │
│ varchar │         int64          │
├─────────┼────────────────────────┤
│ 1750153 │                      2 │
│ 1173514 │                      2 │
│ 879526  │                      2 │
│ 1563568 │                      2 │
│ 1784168 │                      2 │
│ 1822928 │                      2 │
│ 714284  │                      2 │
│ 1912582 │                      2 │
│ 1829794 │                      2 │
│ 1707919 │                      2 │
│    ·    │                      · │
│    ·    │                      · │
│    ·    │                      · │
│ 1758057 │                      2 │
│ 1499961 │                      2 │
│ 1818644 │                      2 │
│ 1284454 │                      3 │
│ 1498233 │                      2 │
│ 1759546 │                      2 │
│ 1722969 │                      2 │
│ 743238  │                      2 │
│ 1805521 │                      2 │
│ 1794621 │                      2 │
└

In [7]:
conn.sql("""
         SELECT name, COUNT(DISTINCT cik)
         FROM sub
         GROUP BY name
         HAVING COUNT(DISTINCT cik) > 1
         """)

┌─────────┬─────────────────────┐
│  name   │ count(DISTINCT cik) │
│ varchar │        int64        │
└─────────┴─────────────────────┘
             0 rows            

In [14]:
conn.sql(""" 
         SELECT count(distinct cik)
         FROM sub
         WHERE changed is not null""").df()

,count(DISTINCT cik)
0,92


Some companies out of the 92, had their name change before 2019, and no name change even once after that. There are 35 such companies that had their name change more than once after 2019 and so have count of distinct names more than 1.

In [15]:
conn.execute("""
             CREATE SCHEMA IF NOT EXISTS clean;""")

In [22]:
conn.sql("""
         CREATE TABLE clean.sub AS
         SELECT 
         adsh,
         cik,
         name,
         CAST(sic AS INTEGER) AS sic,
         strptime(changed, '%Y%m%d')::DATE AS changed,
         afs,
         wksi,
         CAST(fye AS INTEGER) AS fye,
         form,
         strptime(period, '%Y%m%d')::DATE AS period,
         CAST(fy AS INTEGER) AS fy,
         fp,
         strptime(filed, '%Y%m%d')::DATE AS filed,
         CAST(accepted AS TIMESTAMP) AS accepted,
         CAST(prevrpt AS INTEGER) AS prevrpt,
         CAST(nciks AS INTEGER) AS nciks
         FROM main.sub
         """)

In [39]:
conn.sql("""
         SUMMARIZE clean.sub""").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,adsh,VARCHAR,0000014195-19-000008,0001968915-25-000066,2724,None,None,None,None,None,2578,0.0
1,cik,VARCHAR,1005229,97216,145,None,None,None,None,None,2578,0.0
2,name,VARCHAR,10X CAPITAL VENTURE ACQUISITION CORP,YONG BAI CHAO NEW RETAIL CORP,195,None,None,None,None,None,2578,0.0
3,sic,INTEGER,3510,3716,10,3663.8366951124904,83.87305911935593,3537,3713,3714,2578,0.0
4,fye,INTEGER,228,1231,12,1119.1602017067494,247.49336200432322,1231,1231,1231,2578,0.0
5,form,VARCHAR,10-K,10-Q,2,None,None,None,None,None,2578,0.0
6,period,DATE,2017-12-31,2025-11-30,82,2022-05-29 23:41:00.512025,None,2020-09-30,2022-06-26,2024-01-31,2578,0.0
7,fy,INTEGER,2017,2026,11,2021.867726920093,2.0135257497248333,2020,2022,2024,2578,0.0
8,fp,VARCHAR,FY,Q3,4,None,None,None,None,None,2578,0.0
9,filed,DATE,2019-01-08,2025-12-19,803,2022-07-17 23:52:10.799069,None,2020-11-05,2022-07-30,2024-04-16,2578,0.0


In [26]:
conn.sql(""" 
         USE clean;
         SELECT form, COUNT(*)
         FROM sub
         GROUP BY form""").df()

,form,count_star()
0,10-Q,1944
1,10-K,634


In [30]:
conn.sql("""
         SELECT *
         FROM clean.sub
        WHERE len(adsh) != len(TRIM(adsh)) OR len(cik)!=len(TRIM(cik)) OR len(name)!=len(TRIM(name)) OR len(fp)!=len(TRIM(fp)) OR len(afs)!=len(TRIM(afs)) OR len(wksi)!=len(TRIM(wksi)) OR len(form) != len(TRIM(form)) """).df()

,adsh,cik,name,sic,changed,afs,wksi,fye,form,period,fy,fp,filed,accepted,prevrpt,nciks


In [38]:
conn.sql("""
    CREATE OR REPLACE TABLE clean.sub AS 
    SELECT * EXCLUDE (changed, afs, wksi, accepted, nciks) 
    FROM clean.sub;
""")


In [1]:
conn.close()

NameError: name 'conn' is not defined